# Capítulo 28 · Laboratório 28: stops, o seguro que mantém você no jogo
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005, Ibovespa desde 1995 e S&P 500 desde 1928). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap28.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Custo de 0,10% por operação; o stop executa no nível ou na abertura, se o papel abrir abaixo (gap). A limpeza da Parte 1 corrige três tipos de dado sujo (máxima impossível, desdobramento sem ajuste e meses sem cotação); a Parte 5 mostra o que foi corrigido. O sorteio das entradas usa semente fixa: os números se repetem a cada execução.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM, CUSTO, PRAZO = "2026-10-01", 0.001, 20
CACHE = "dados_cap28.pkl"
rng = np.random.default_rng(30)            # sorteio das entradas de controle (semente fixa)


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, auto_adjust=True, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, indices = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, group_by="ticker")
    indices = {k: baixa(k, start=s, end=FIM, multi_level_index=False)
               for k, s in (("^BVSP", "1995-01-01"), ("^GSPC", "1927-01-01"))}
    pd.to_pickle((dados, indices), CACHE)
mostra("Parte 1. Dados: 40 ações desde 2005 (SUZB3 desde 10/11/2017), Ibovespa e S&P 500, até 30/09/2026")


CORRIGIDAS = []                            # o que a limpeza mudou (registro da Parte 5)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.append(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}")
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.append(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido")
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.append(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data")
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.append(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}")
        d = d.loc[ini:]
    return d


ERROS_INDICE = {"^GSPC": ["1935-04-16"]}   # 8,22 entre 9,13 e 9,01, sem máxima nem mínima: erro do provedor (Parte 5)


def atr(h, l, c, n):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def sai_no_stop(O, L, k, stop):
    """Stop de venda atingido na barra k: executa no stop, ou na abertura se ela já abriu abaixo (gap)."""
    preco = min(stop, O[k])
    return preco, O[k] < stop


def operar(d, i, metodo):
    """Entrada no fechamento da barra i. Devolve (retorno, dias, stopado, via_gap, nivel_stop, k)."""
    O, H, L, C = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close"))
    A20, A22 = d["atr20"].to_numpy(), d["atr22"].to_numpy()
    e, n = C[i], len(C)
    if metodo in ("sem stop", "fixo 8%", "2 ATR", "3 ATR", "estrutural (mínima de 10 dias)"):
        stop = {"sem stop": -np.inf, "fixo 8%": e * 0.92, "2 ATR": e - 2 * A20[i], "3 ATR": e - 3 * A20[i],
                "estrutural (mínima de 10 dias)": L[i - 9:i + 1].min() * 0.999}[metodo]
        for k in range(i + 1, min(i + PRAZO + 1, n)):
            if L[k] <= stop:
                p, g = sai_no_stop(O, L, k, stop)
                return p / e - 1, k - i, True, g, stop, k
        k = min(i + PRAZO, n - 1)
        return C[k] / e - 1, k - i, False, False, stop, k
    # stops móveis: ficam até serem atingidos (ou 250 pregões)
    topo, stop = H[i], None
    af, ep = 0.02, H[i]
    sar = L[max(i - 4, 0):i + 1].min()
    for k in range(i + 1, min(i + 251, n)):
        if metodo == "móvel: chandelier 3 ATR(22)":
            stop = topo - 3 * A22[k - 1]
        elif metodo == "móvel: mínima de 10 dias":
            stop = L[k - 10:k].min()
        else:                                           # SAR parabólico de Wilder, só comprado
            stop = sar
        if L[k] <= stop:
            p, g = sai_no_stop(O, L, k, stop)
            return p / e - 1, k - i, True, g, stop, k
        topo = max(topo, H[k])
        if H[k] > ep:
            ep, af = H[k], min(af + 0.02, 0.2)
        sar = min(sar + af * (ep - sar), L[k], L[k - 1])
    k = min(i + 250, n - 1)
    return C[k] / e - 1, k - i, False, False, stop, k

## Parte 2: o mesmo sinal, oito saídas

In [ ]:
METODOS = ["sem stop", "fixo 8%", "2 ATR", "3 ATR", "estrutural (mínima de 10 dias)",
           "móvel: chandelier 3 ATR(22)", "móvel: mínima de 10 dias", "móvel: SAR parabólico"]
linhas = []
for t in TICKERS:
    d = limpa(dados[t], t)
    if t == "SUZB3.SA":
        d = d.loc["2017-11-10":]                     # antes disso, preço parado no Yahoo (Capítulo 26)
    c = d["Close"]
    d["atr20"], d["atr22"] = atr(d["High"], d["Low"], c, 20), atr(d["High"], d["Low"], c, 22)
    rompe = ((c > d["High"].rolling(20).max().shift(1)) & (c > c.rolling(200).mean())).to_numpy()
    validos = np.arange(210, len(d) - 30)
    sinais = {"rompimento de 20 dias": [i for i in validos if rompe[i]]}
    sinais["entrada sorteada"] = sorted(rng.choice(validos, size=len(sinais["rompimento de 20 dias"]), replace=False))
    C = c.to_numpy()
    for nome_s, idx in sinais.items():
        livre = 0
        for i in idx:
            if i < livre:                              # uma posição por vez em cada ação (no sinal de rompimento)
                continue
            for m in METODOS:
                r, dias, st, gap, nivel, k = operar(d, i, m)
                ref = C[i:k + 1].max()                     # maior fechamento enquanto estava posicionado
                volta = bool(st and k + 11 < len(C) and (C[k + 1:k + 11] > ref).any())
                extra = (1 + r) / (nivel / C[i]) - 1 if st else np.nan       # execução abaixo do nível do stop
                linhas.append((nome_s, m, t, r - CUSTO, dias, st, gap, extra, volta, d.index[k]))
            livre = i + PRAZO
R = pd.DataFrame(linhas, columns=["sinal", "metodo", "ativo", "r", "dias", "stopado", "gap", "extra", "volta", "saida"])

mostra("Parte 2. O mesmo sinal, saídas diferentes (40 ações, 2005 a 30/09/2026, custo de 0,10% por operação)")
for s, g in R.groupby("sinal", sort=False):
    mostra(f"\n   Sinal: {s} ({g['metodo'].eq('sem stop').sum()} operações por método)")
    mostra("   saída                              média | acerto | dias | média/dia | 5% piores | pior    | stopadas | violinadas*")
    for m in METODOS:
        x = g[g["metodo"] == m]
        cauda = x["r"].quantile(0.05)
        st = x["stopado"]
        mostra(f"   {m:<32} {x['r'].mean():>+6.2%} | {(x['r'] > 0).mean():>5.0%} | {x['dias'].mean():>4.1f} |"
              f" {x['r'].mean() / x['dias'].mean():>+8.3%} | {x.loc[x['r'] <= cauda, 'r'].mean():>+8.1%} |"
              f" {x['r'].min():>+6.0%} | {st.mean():>7.0%} | {x.loc[st, 'volta'].mean() if st.any() else 0:>7.0%}")
mostra("   * stopadas em que o preço superou, nos 10 pregões seguintes, o maior fechamento da operação (a tendência seguiu sem você)")

## Parte 3: o stop não garante o preço

In [ ]:
S = R[R["stopado"]]
mostra("\nParte 3. O stop não garante o preço (todas as saídas por stop da Parte 2)")
for m in METODOS[1:]:
    x = S[S["metodo"] == m]
    if len(x):
        xg = x[x["gap"]]
        mostra(f"   {m:<32} stops {len(x):>6} | executados abaixo do nível por gap: {x['gap'].mean():>4.0%}"
              f" | perda extra média nesses casos {xg['extra'].mean():+.1%}")
g = S[S["gap"]]
j = g["extra"].idxmin()
mostra(f"   1% piores gaps: abertura {g['extra'].quantile(0.01):+.0%} ou mais abaixo do stop; pior: {g.loc[j, 'extra']:+.0%}"
      f" ({g.loc[j, 'ativo']}, {g.loc[j, 'saida']:%d/%m/%Y})")

In [ ]:
import matplotlib.pyplot as plt
g = R[R["sinal"] == "rompimento de 20 dias"]
ordem = ["sem stop", "fixo 8%", "3 ATR", "móvel: chandelier 3 ATR(22)"]
plt.figure(figsize=(8, 4))
plt.boxplot([g.loc[g["metodo"] == m, "r"] * 100 for m in ordem], whis=(5, 95), showfliers=False)
plt.xticks(range(1, len(ordem) + 1), ordem, fontsize=8)
plt.axhline(0, color="gray", lw=0.6)
plt.ylabel("resultado por operação (%), do 5º ao 95º percentil")
plt.show()

## Parte 4: crashes

In [ ]:
mostra("\nParte 4. Crashes")
for nome, k, ini in (("Ibovespa", "^BVSP", "1995-07-01"), ("S&P 500", "^GSPC", "1928-01-01")):
    x = indices[k]["Close"].dropna().loc[ini:]
    x = x.drop(pd.to_datetime(ERROS_INDICE.get(k, [])), errors="ignore")   # erro de dado sai da contagem
    r = x.pct_change().dropna()
    mostra(f"\n   {nome}, desde {x.index[0]:%m/%Y}: dias com queda de 5% ou mais: {(r <= -0.05).sum()}"
          f" | de 8% ou mais: {(r <= -0.08).sum()} | de 10% ou mais: {(r <= -0.10).sum()}")
    mostra("   piores dias: " + "; ".join(f"{d:%d/%m/%Y} {v:+.1%}" for d, v in r.nsmallest(6).items()))
    topo = x.cummax()
    dd = x / topo - 1
    eventos, i = [], 0
    vals, idx = dd.to_numpy(), dd.index
    while i < len(vals):
        if vals[i] < 0:
            j = i
            while j < len(vals) and vals[j] < 0:
                j += 1
            seg = dd.iloc[i:j]
            fundo = seg.idxmin()
            ini_t = idx[i - 1] if i > 0 else idx[0]
            eventos.append((seg.min(), ini_t, fundo, idx[j] if j < len(vals) else None))
            i = j
        else:
            i += 1
    E = sorted(eventos)[:5]
    mostra("   maiores quedas (topo -> fundo, volta ao topo):")
    for q, a, f, v in E:
        anos_v = f"{(v - a).days / 365.25:.1f} anos" if v is not None else "ainda não voltou"
        mostra(f"      {q:+.0%}  {a:%m/%Y} -> {f:%m/%Y}, volta: {v:%m/%Y} ({anos_v})" if v is not None
              else f"      {q:+.0%}  {a:%m/%Y} -> {f:%m/%Y}, {anos_v}")

ib = indices["^BVSP"]
f20 = ib.loc["2020-02-21", "Close"]
jan = ib.loc["2020-01-01":"2020-02-21", "Close"]
m = ib.loc["2020-02-21":"2020-04-30"]
mostra(f"\n   Covid no Ibovespa: fechamento de 21/02/2020 {f20:_.0f}; topo de janeiro {jan.max():_.0f} ({jan.idxmax():%d/%m});"
      f" menor fechamento {m['Close'].min():_.0f} ({m['Close'].idxmin():%d/%m}, {m['Close'].min() / f20 - 1:+.0%} desde 21/02);"
      f" menor mínima {m['Low'].min():_.0f} ({m['Low'].idxmin():%d/%m}, {m['Low'].min() / f20 - 1:+.0%})")
sp = indices["^GSPC"]["Close"]
pico = sp.loc["2020-02-01":"2020-02-29"].max()
volta = sp.loc["2020-03-01":][sp.loc["2020-03-01":] >= pico].index[0]
mostra(f"   S&P 500 voltou ao topo de fevereiro de 2020 em {volta:%d/%m/%Y}")

## Parte 5: dados a conferir

In [ ]:
mostra("\nParte 5. Dados a conferir antes de acreditar")
mostra("   S&P 500 em abril de 1935: o dia 16 aparece com queda de 10% e volta no dia seguinte (fora da contagem):")
mostra(indices["^GSPC"].loc["1935-04-11":"1935-04-22", ["Open", "High", "Low", "Close"]].round(2).to_string())
mostra(f"   ações: {len(CORRIGIDAS)} correções da limpeza:")
for linha in CORRIGIDAS:
    mostra("      " + linha)

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `sai_no_stop` e `operar` da Parte 1. Mostre por que `min(stop, O[k])` faz o stop sair na abertura quando há gap e por que os stops móveis usam só barras anteriores à barra da saída (`A22[k - 1]` e `L[k - 10:k]`)." Confira com a Parte 3, que imprime a porcentagem de stops executados abaixo do nível por gap em cada saída (de 13% a 16%) e o pior caso, a BBAS3 em 18/05/2017.
2. **Adaptar:** "Numa célula nova, depois da Parte 3, baixe [ATIVO] com a função `baixa` (com `multi_level_index=False`), passe por `limpa` e rode as saídas da lista `METODOS` sobre o mesmo sinal da Parte 2: rompimento da máxima de 20 dias com o preço acima da média de 200. Não mude a fórmula do ATR, os múltiplos nem o prazo." Antes de olhar o ativo novo, confira se a Parte 2 continua mostrando 3.789 operações por método e +1,04% sem stop, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque os múltiplos de ATR de 2 e 3 por 1,5 e 4. A média muda? E a cauda dos 5% piores?
2. Na Parte 2, acrescente uma saída por tempo de 10 pregões. Ela corta a cauda tanto quanto o stop?
3. Na Parte 3, conte quantos gaps aconteceram na segunda-feira. Fim de semana pesa mais?
4. Na Parte 4, troque o limite de queda de 5% por 3% e compare o número de dias do Ibovespa e do S&P 500.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.